串口连接

In [23]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EL-A3 机械臂串口检测与连接工具 (Linux)
适用于 CH340 USB-CAN 适配器
"""

import os
import sys
import time
import glob
import subprocess
import serial
import serial.tools.list_ports

print("=" * 60)
print("EL-A3 机械臂串口检测与连接工具")
print("=" * 60)
# 检查操作系统
print(f"\n【系统信息】")
print(f"操作系统: {os.name}")
print(f"平台: {sys.platform}")

# 检查是否为 Linux
is_linux = sys.platform.startswith('linux')
print(f"是否为 Linux: {is_linux}")

EL-A3 机械臂串口检测与连接工具

【系统信息】
操作系统: posix
平台: linux
是否为 Linux: True


In [2]:
from el_a3_sdk import ELA3Interface

arm = ELA3Interface(can_name="can0")
arm.ConnectPort()
arm.EnableArm()

# 直接关节控制
#arm.JointCtrl(0.0, 0.5, -0.3, 0.0, 0.0, 0.0)

# 读取反馈
print(arm.GetArmJointMsgs())
print(arm.GetArmEndPoseMsgs())

arm.DisableArm()
arm.DisconnectPort()

ArmJointStates(joint_1=-0.05274662394140606, joint_2=0.00019180590524214836, joint_3=-0.013618219272144572, joint_4=0.30899931334401387, joint_5=-6.317511100938429, joint_6=0.13944289311055158, joint_7=6.30216662851911, timestamp=1776850569.6120095, hz=0.0)
ArmEndPose(x=-0.11423546522362388, y=0.0031635464658733544, z=0.10298372127745842, rx=1.2730670403686168, ry=-0.12338315650613228, rz=1.5916791503397245, timestamp=0.0)


In [46]:
# Python pyserial 示例
import serial
ser = serial.Serial('/dev/ttyUSB0', 115200)

Looking in indexes: https://mirrors.aliyun.com/pypi/simple


In [42]:
import can
bus = can.interface.Bus(bustype='serial', channel='/dev/ttyUSB0', bitrate=1000000)

/tmp/ipykernel_98111/2827664523.py:2: DeprecationWarning: The 'bustype' argument is deprecated since python-can v4.2.0, and scheduled for removal in python-can v5.0.0. Use 'interface' instead.
  bus = can.interface.Bus(bustype='serial', channel='/dev/ttyUSB0', bitrate=1000000)


In [44]:
import can
bus = can.interface.Bus(interface='slcan', channel='/dev/ttyUSB0', bitrate=1000000)

slcanBus was not properly shut down


In [24]:
print(f"\n【查找可用串口】")

def find_serial_ports():
    """查找所有可用串口"""
    ports = []
    
    if sys.platform == 'win32':
        # Windows: 使用 pyserial 列出所有 COM 口
        port_list = list(serial.tools.list_ports.comports())
        for port in port_list:
            ports.append({
                'device': port.device,
                'description': port.description,
                'hwid': port.hwid
            })
    else:
        # Linux: 查找 /dev/ttyUSB* 和 /dev/ttyACM*
        # USB 转串口 (CH340/CP2102/FT232等)
        usb_ports = glob.glob('/dev/ttyUSB*')
        for p in usb_ports:
            ports.append({
                'device': p,
                'description': 'USB Serial Port',
                'hwid': 'USB'
            })
        
        # ACM 设备 (部分 USB-CAN 适配器)
        acm_ports = glob.glob('/dev/ttyACM*')
        for p in acm_ports:
            ports.append({
                'device': p,
                'description': 'ACM Serial Port',
                'hwid': 'ACM'
            })
        
        # 系统串口
        sys_ports = glob.glob('/dev/ttyS*')
        for p in sys_ports:
            ports.append({
                'device': p,
                'description': 'System Serial Port',
                'hwid': 'SYS'
            })
    
    return ports

ports = find_serial_ports()

if not ports:
    print("⚠️ 未找到任何串口设备")
    print("请检查：")
    print("  1. USB 设备是否已插入")
    print("  2. 驱动是否正确加载")
else:
    print(f"找到 {len(ports)} 个串口:")
    for i, port in enumerate(ports):
        print(f"  {i+1}. {port['device']} - {port['description']}")
        if 'hwid' in port:
            print(f"     HWID: {port['hwid']}")


【查找可用串口】
找到 32 个串口:
  1. /dev/ttyS0 - System Serial Port
     HWID: SYS
  2. /dev/ttyS1 - System Serial Port
     HWID: SYS
  3. /dev/ttyS2 - System Serial Port
     HWID: SYS
  4. /dev/ttyS3 - System Serial Port
     HWID: SYS
  5. /dev/ttyS4 - System Serial Port
     HWID: SYS
  6. /dev/ttyS5 - System Serial Port
     HWID: SYS
  7. /dev/ttyS6 - System Serial Port
     HWID: SYS
  8. /dev/ttyS7 - System Serial Port
     HWID: SYS
  9. /dev/ttyS8 - System Serial Port
     HWID: SYS
  10. /dev/ttyS9 - System Serial Port
     HWID: SYS
  11. /dev/ttyS10 - System Serial Port
     HWID: SYS
  12. /dev/ttyS11 - System Serial Port
     HWID: SYS
  13. /dev/ttyS12 - System Serial Port
     HWID: SYS
  14. /dev/ttyS13 - System Serial Port
     HWID: SYS
  15. /dev/ttyS14 - System Serial Port
     HWID: SYS
  16. /dev/ttyS15 - System Serial Port
     HWID: SYS
  17. /dev/ttyS16 - System Serial Port
     HWID: SYS
  18. /dev/ttyS17 - System Serial Port
     HWID: SYS
  19. /dev/ttyS18 - System

In [25]:
if is_linux:
    print(f"\n【检查 CH340/CH341 驱动状态】")
    
    # 检查内核模块
    result = subprocess.run(['lsmod'], capture_output=True, text=True)
    if 'ch34' in result.stdout:
        print("✅ CH341/CH340 驱动已加载")
        # 显示详细信息
        lines = [l for l in result.stdout.split('\n') if 'ch34' in l]
        for line in lines:
            print(f"   {line}")
    else:
        print("❌ CH341/CH340 驱动未加载")
        print("   尝试加载驱动...")
        os.system('sudo modprobe ch341')
        os.system('sudo modprobe usbserial')
        time.sleep(1)
        # 重新检查
        result = subprocess.run(['lsmod'], capture_output=True, text=True)
        if 'ch34' in result.stdout:
            print("✅ 驱动加载成功")
        else:
            print("❌ 驱动加载失败，请检查内核版本")


【检查 CH340/CH341 驱动状态】
✅ CH341/CH340 驱动已加载
   ch341                  24576  0
   usbserial              69632  1 ch341


In [26]:
if is_linux:
    print(f"\n【检查 USB 设备识别】")
    
    # 使用 lsusb 查看 USB 设备
    result = subprocess.run(['lsusb'], capture_output=True, text=True)
    
    # 查找 CH340 相关设备
    ch340_lines = [l for l in result.stdout.split('\n') if '1a86' in l or 'CH34' in l or 'QinHeng' in l]
    
    if ch340_lines:
        print("✅ 发现 CH340 USB 设备:")
        for line in ch340_lines:
            print(f"   {line}")
    else:
        print("⚠️ 未在 lsusb 中发现 CH340 设备")
        print("   所有 USB 设备:")
        for line in result.stdout.split('\n')[:10]:
            if line.strip():
                print(f"   {line}")


【检查 USB 设备识别】
⚠️ 未在 lsusb 中发现 CH340 设备
   所有 USB 设备:
   Bus 006 Device 002: ID 174c:55aa ASMedia Technology Inc. ASM1051E SATA 6Gb/s bridge, ASM1053E SATA 6Gb/s bridge, ASM1153 SATA 3Gb/s bridge, ASM1153E SATA 6Gb/s bridge
   Bus 006 Device 001: ID 1d6b:0003 Linux Foundation 3.0 root hub
   Bus 005 Device 002: ID 2b89:5259 EVISION UGREEN Keyboard
   Bus 005 Device 001: ID 1d6b:0002 Linux Foundation 2.0 root hub
   Bus 004 Device 001: ID 1d6b:0003 Linux Foundation 3.0 root hub
   Bus 003 Device 003: ID 0b05:18f3 ASUSTek Computer, Inc. AURA LED Controller
   Bus 003 Device 004: ID 8087:0025 Intel Corp. Wireless-AC 9260 Bluetooth Adapter
   Bus 003 Device 001: ID 1d6b:0002 Linux Foundation 2.0 root hub
   Bus 002 Device 001: ID 1d6b:0003 Linux Foundation 3.0 root hub
   Bus 001 Device 002: ID 046d:c077 Logitech, Inc. M105 Optical Mouse


In [28]:
print(f"\n【测试串口连接】")

def test_serial_port(port_name, baudrate=921600, timeout=1):
    """测试串口是否可以打开"""
    try:
        ser = serial.Serial(
            port=port_name,
            baudrate=baudrate,
            bytesize=serial.EIGHTBITS,
            parity=serial.PARITY_NONE,
            stopbits=serial.STOPBITS_ONE,
            timeout=timeout
        )
        
        if ser.is_open:
            print(f"✅ 成功打开 {port_name}")
            print(f"   波特率: {baudrate}")
            print(f"   状态: {ser}")
            
            # 尝试读取数据（如果有）
            time.sleep(0.1)
            waiting = ser.in_waiting
            print(f"   缓冲区数据: {waiting} bytes")
            
            ser.close()
            print(f"   已关闭连接")
            return True
        else:
            print(f"❌ 无法打开 {port_name}")
            return False
            
    except serial.SerialException as e:
        print(f"❌ 串口错误: {e}")
        return False
    except PermissionError as e:
        print(f"❌ 权限错误: {e}")
        print(f"   请检查当前用户是否在 dialout 组")
        print(f"   运行: sudo usermod -a -G dialout $USER")
        return False
    except Exception as e:
        print(f"❌ 未知错误: {e}")
        return False

# 测试找到的所有串口
for port_info in ports:
    port_device = port_info['device']
    print(f"\n测试 {port_device}...")
    test_serial_port(port_device)


【测试串口连接】

测试 /dev/ttyS0...
✅ 成功打开 /dev/ttyS0
   波特率: 921600
   状态: Serial<id=0x7ea8a3bb3d90, open=True>(port='/dev/ttyS0', baudrate=921600, bytesize=8, parity='N', stopbits=1, timeout=1, xonxoff=False, rtscts=False, dsrdtr=False)
   缓冲区数据: 0 bytes
   已关闭连接

测试 /dev/ttyS1...
❌ 串口错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS2...
❌ 串口错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS3...
❌ 串口错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS4...
❌ 串口错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS5...
❌ 串口错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS6...
❌ 串口错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS7...
❌ 串口错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS8...
❌ 串口错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS9...
❌ 串口错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS10...
❌ 串口错误: Could not con

In [29]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
修复 CH340 设备未生成 ttyUSB 的问题
"""

import os
import subprocess
import time
import glob

print("=" * 60)
print("修复 CH340 设备绑定问题")
print("=" * 60)

修复 CH340 设备绑定问题


In [30]:
print("\n【1. 检查 USB 设备状态】")

# 查看详细 USB 信息
result = subprocess.run(['lsusb', '-d', '1a86:7523', '-v'], 
                       capture_output=True, text=True)
print("CH340 设备详情:")
print(result.stdout[:2000] if len(result.stdout) > 2000 else result.stdout)

# 检查 sysfs 中的 USB 设备
print("\n检查 USB 设备树:")
result = subprocess.run(['lsusb', '-t'], capture_output=True, text=True)
print(result.stdout)


【1. 检查 USB 设备状态】
CH340 设备详情:


检查 USB 设备树:
/:  Bus 06.Port 1: Dev 1, Class=root_hub, Driver=xhci_hcd/4p, 10000M
    |__ Port 2: Dev 2, If 0, Class=Mass Storage, Driver=uas, 5000M
/:  Bus 05.Port 1: Dev 1, Class=root_hub, Driver=xhci_hcd/4p, 480M
    |__ Port 1: Dev 2, If 0, Class=Human Interface Device, Driver=usbhid, 12M
    |__ Port 1: Dev 2, If 1, Class=Human Interface Device, Driver=usbhid, 12M
/:  Bus 04.Port 1: Dev 1, Class=root_hub, Driver=xhci_hcd/4p, 10000M
/:  Bus 03.Port 1: Dev 1, Class=root_hub, Driver=xhci_hcd/6p, 480M
    |__ Port 5: Dev 4, If 0, Class=Wireless, Driver=btusb, 12M
    |__ Port 5: Dev 4, If 1, Class=Wireless, Driver=btusb, 12M
    |__ Port 6: Dev 3, If 0, Class=Vendor Specific Class, Driver=, 12M
    |__ Port 6: Dev 3, If 2, Class=Human Interface Device, Driver=usbhid, 12M
/:  Bus 02.Port 1: Dev 1, Class=root_hub, Driver=xhci_hcd/4p, 10000M
/:  Bus 01.Port 1: Dev 1, Class=root_hub, Driver=xhci_hcd/6p, 480M
    |__ Port 2: Dev 2, If 0, Class=Human Interface

In [31]:
print("\n【2. 检查设备绑定状态】")

# 查看 /sys/bus/usb/devices 下的设备
usb_devices = glob.glob('/sys/bus/usb/devices/*/idVendor')
print(f"找到 {len(usb_devices)} 个 USB 设备")

for dev_path in usb_devices:
    vendor_path = dev_path
    product_path = dev_path.replace('idVendor', 'idProduct')
    
    if os.path.exists(vendor_path) and os.path.exists(product_path):
        with open(vendor_path, 'r') as f:
            vendor = f.read().strip()
        with open(product_path, 'r') as f:
            product = f.read().strip()
        
        if vendor == '1a86' and product == '7523':
            print(f"\n🎯 找到 CH340 设备:")
            print(f"   路径: {os.path.dirname(dev_path)}")
            
            # 检查是否有 tty 设备
            dev_dir = os.path.dirname(dev_path)
            tty_dirs = glob.glob(f"{dev_dir}/*/tty*")
            if tty_dirs:
                print(f"   TTY 设备: {tty_dirs}")
            else:
                print(f"   ⚠️ 无 TTY 设备绑定")
                
            # 检查驱动绑定
            driver_path = f"{dev_dir}/driver"
            if os.path.exists(driver_path):
                driver = os.readlink(driver_path)
                print(f"   驱动: {driver}")


【2. 检查设备绑定状态】
找到 11 个 USB 设备


In [32]:
print("\n【3. 强制重新绑定 CH340 驱动】")

# 先卸载驱动
print("卸载 ch341 驱动...")
os.system('sudo modprobe -r ch341 2>/dev/null')
os.system('sudo modprobe -r usbserial 2>/dev/null')
time.sleep(1)

# 重新加载
print("重新加载驱动...")
os.system('sudo modprobe usbserial')
time.sleep(0.5)
os.system('sudo modprobe ch341')
time.sleep(2)

# 检查是否生成 ttyUSB
print("\n检查生成的设备:")
ttyusb_devices = glob.glob('/dev/ttyUSB*')
if ttyusb_devices:
    print(f"✅ 成功生成: {ttyusb_devices}")
    for dev in ttyusb_devices:
        # 设置权限
        os.system(f'sudo chmod 666 {dev}')
        print(f"   已设置权限: {dev}")
else:
    print("❌ 仍未生成 /dev/ttyUSB*")


【3. 强制重新绑定 CH340 驱动】
卸载 ch341 驱动...
重新加载驱动...


sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: 需要密码
sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: 需要密码



检查生成的设备:
❌ 仍未生成 /dev/ttyUSB*


In [33]:
print("\n【5. 创建 udev 规则（永久修复）】")

udev_rule = '''# CH340 USB to Serial
SUBSYSTEM=="usb", ATTR{idVendor}=="1a86", ATTR{idProduct}=="7523", ATTR{manufacturer}=="QinHeng Electronics", MODE="0666", GROUP="dialout"

# 强制绑定到 usbserial
SUBSYSTEM=="usb", ATTR{idVendor}=="1a86", ATTR{idProduct}=="7523", RUN+="/sbin/modprobe ch341"
'''

# 写入规则文件
rule_file = '/etc/udev/rules.d/99-ch340.rules'
print(f"创建 udev 规则: {rule_file}")

with open('/tmp/99-ch340.rules', 'w') as f:
    f.write(udev_rule)

os.system(f'sudo cp /tmp/99-ch340.rules {rule_file}')
os.system('sudo udevadm control --reload-rules')
os.system('sudo udevadm trigger')

print("✅ udev 规则已创建")
print("   重新插拔 USB 设备后生效")




【5. 创建 udev 规则（永久修复）】
创建 udev 规则: /etc/udev/rules.d/99-ch340.rules
✅ udev 规则已创建
   重新插拔 USB 设备后生效


sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: 需要密码
sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: 需要密码
sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: 需要密码


In [34]:
print("\n【6. 测试最终连接】")

import serial

# 测试所有可能的端口
test_ports = glob.glob('/dev/ttyUSB*') + glob.glob('/dev/ttyACM*')

if not test_ports:
    # 如果还是没有，尝试使用 /dev/ttyS0 或其他
    print("⚠️ 未找到标准 USB 串口，测试系统串口...")
    test_ports = ['/dev/ttyS0', '/dev/ttyS1', '/dev/ttyS2']

print(f"测试端口: {test_ports}")

for port in test_ports:
    if not os.path.exists(port):
        continue
        
    print(f"\n测试 {port}...")
    try:
        ser = serial.Serial(port, 921600, timeout=1)
        if ser.is_open:
            print(f"✅ 成功打开 {port}")
            
            # 清空缓冲区
            ser.reset_input_buffer()
            ser.reset_output_buffer()
            
            # 发送 AT 命令测试
            at_cmd = bytes.fromhex("41542B41540D0A")
            ser.write(at_cmd)
            time.sleep(0.2)
            
            # 读取响应
            waiting = ser.in_waiting
            if waiting > 0:
                resp = ser.read(waiting)
                print(f"🎉 收到响应: {resp.hex()}")
                print(f"   这是 EL-A3 设备!")
                working_port = port
            else:
                print(f"   无响应（可能是正确设备但未启动）")
                working_port = port
            
            ser.close()
            
    except Exception as e:
        print(f"❌ 错误: {e}")

print("\n" + "=" * 60)
print("修复完成!")
print("=" * 60)
print("\n如果仍未生成 /dev/ttyUSB0，请尝试:")
print("1. 物理重新插拔 USB 设备")
print("2. 重启系统: sudo reboot")
print("3. 检查 USB 线是否损坏")
print("4. 尝试其他 USB 端口")


【6. 测试最终连接】
⚠️ 未找到标准 USB 串口，测试系统串口...
测试端口: ['/dev/ttyS0', '/dev/ttyS1', '/dev/ttyS2']

测试 /dev/ttyS0...
✅ 成功打开 /dev/ttyS0
   无响应（可能是正确设备但未启动）

测试 /dev/ttyS1...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS2...
❌ 错误: Could not configure port: (5, 'Input/output error')

修复完成!

如果仍未生成 /dev/ttyUSB0，请尝试:
1. 物理重新插拔 USB 设备
2. 重启系统: sudo reboot
3. 检查 USB 线是否损坏
4. 尝试其他 USB 端口


In [35]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EL-A3 连接测试 - 无需 sudo 版本
假设驱动已加载，设备已生成
"""

import os
import glob
import time
import serial

print("=" * 60)
print("EL-A3 连接测试")
print("=" * 60)
print("\n【检查串口设备】")

# 查找所有可能的串口
ports = glob.glob('/dev/ttyUSB*') + glob.glob('/dev/ttyACM*')
print(f"找到 USB 串口: {ports}")

# 如果没有 ttyUSB，检查 ttyS
if not ports:
    ports = glob.glob('/dev/ttyS*')
    print(f"找到系统串口: {ports}")

# 检查权限
for p in ports:
    if os.path.exists(p):
        stat = os.stat(p)
        print(f"  {p}: 权限={oct(stat.st_mode)[-3:]}, 所有者={stat.st_uid}")

EL-A3 连接测试

【检查串口设备】
找到 USB 串口: []
找到系统串口: ['/dev/ttyS0', '/dev/ttyS1', '/dev/ttyS2', '/dev/ttyS3', '/dev/ttyS4', '/dev/ttyS5', '/dev/ttyS6', '/dev/ttyS7', '/dev/ttyS8', '/dev/ttyS9', '/dev/ttyS10', '/dev/ttyS11', '/dev/ttyS12', '/dev/ttyS13', '/dev/ttyS14', '/dev/ttyS15', '/dev/ttyS16', '/dev/ttyS17', '/dev/ttyS18', '/dev/ttyS19', '/dev/ttyS20', '/dev/ttyS21', '/dev/ttyS22', '/dev/ttyS23', '/dev/ttyS24', '/dev/ttyS25', '/dev/ttyS26', '/dev/ttyS27', '/dev/ttyS28', '/dev/ttyS29', '/dev/ttyS30', '/dev/ttyS31']
  /dev/ttyS0: 权限=660, 所有者=0
  /dev/ttyS1: 权限=660, 所有者=0
  /dev/ttyS2: 权限=660, 所有者=0
  /dev/ttyS3: 权限=660, 所有者=0
  /dev/ttyS4: 权限=660, 所有者=0
  /dev/ttyS5: 权限=660, 所有者=0
  /dev/ttyS6: 权限=660, 所有者=0
  /dev/ttyS7: 权限=660, 所有者=0
  /dev/ttyS8: 权限=660, 所有者=0
  /dev/ttyS9: 权限=660, 所有者=0
  /dev/ttyS10: 权限=660, 所有者=0
  /dev/ttyS11: 权限=660, 所有者=0
  /dev/ttyS12: 权限=660, 所有者=0
  /dev/ttyS13: 权限=660, 所有者=0
  /dev/ttyS14: 权限=660, 所有者=0
  /dev/ttyS15: 权限=660, 所有者=0
  /dev/ttyS16: 权限=660, 所有者=0
  /

In [36]:
def test_el_a3(port, baudrate=921600):
    """测试 EL-A3 连接"""
    print(f"\n测试 {port} @ {baudrate}...")
    
    try:
        ser = serial.Serial(port, baudrate, timeout=1)
        print(f"✅ 打开成功")
        
        # 清空缓冲区
        ser.reset_input_buffer()
        ser.reset_output_buffer()
        
        # 发送 AT 唤醒命令
        at_cmd = bytes.fromhex("41542B41540D0A")  # "AT+AT\r\n"
        ser.write(at_cmd)
        print(f"  发送 AT 命令: {at_cmd.hex()}")
        
        time.sleep(0.2)
        
        # 读取响应
        waiting = ser.in_waiting
        print(f"  缓冲区数据: {waiting} bytes")
        
        if waiting > 0:
            resp = ser.read(waiting)
            print(f"  响应: {resp.hex()}")
            print(f"  🎉 可能是 EL-A3 设备!")
        else:
            print(f"  无响应（设备可能未启动）")
        
        ser.close()
        return True
        
    except PermissionError:
        print(f"❌ 权限不足，需要: sudo chmod 666 {port}")
        return False
    except Exception as e:
        print(f"❌ 错误: {e}")
        return False

# 测试所有端口
for port in ports:
    test_el_a3(port)


测试 /dev/ttyS0 @ 921600...
✅ 打开成功
  发送 AT 命令: 41542b41540d0a
  缓冲区数据: 0 bytes
  无响应（设备可能未启动）

测试 /dev/ttyS1 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS2 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS3 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS4 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS5 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS6 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS7 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS8 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS9 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS10 @ 921600...
❌ 错误: Could not configure port: (5, 'Input/output error')

测试 /dev/ttyS11 @ 921600...
❌ 错误: Could not configure po

In [37]:
print("\n" + "=" * 70)
print("【一键修复命令】")
print("=" * 70)

commands = f'''
# 请在终端中逐行执行以下命令：

# 1. 重新加载驱动
sudo modprobe -r ch341
sudo modprobe -r usbserial
sudo modprobe usbserial
sudo modprobe ch341

# 2. 等待 2 秒
sleep 2

# 3. 强制绑定（如果未自动生成）
echo '{ch340_path.split("/")[-1] if ch340_path else "1-4"}:1.0' | sudo tee /sys/bus/usb/drivers/ch341-uart/bind

# 4. 检查设备
ls -la /dev/ttyUSB*

# 5. 设置权限
sudo chmod 666 /dev/ttyUSB0

# 6. 添加到用户组（永久权限）
sudo usermod -a -G dialout $USER

# 7. 创建 udev 规则（永久修复）
echo 'SUBSYSTEM=="usb", ATTR{{idVendor}}=="1a86", ATTR{{idProduct}}=="7523", MODE="0666", GROUP="dialout"' | sudo tee /etc/udev/rules.d/99-ch340.rules
sudo udevadm control --reload-rules
sudo udevadm trigger

# 8. 重新登录或重启
# 然后重新插拔 USB 设备
'''

print(commands)

# 保存到文件
with open('/tmp/fix_commands.sh', 'w') as f:
    f.write(commands)
print(f"\n✅ 命令已保存到: /tmp/fix_commands.sh")


【一键修复命令】


NameError: name 'ch340_path' is not defined